# Snowflake Semantic View Workshop：投资组合持仓

本 Notebook 按顺序演练 YAML 解析、Snowflake verify-only、受控部署、数据质量检查和指标回归。先在专用 sandbox 中执行 `sql/00_setup_demo.sql`，并检查其中的 `CREATE OR REPLACE TABLE` 语句；不要在有重要数据的 schema 中运行。

示例持仓金额是可重复的教学 fixture，不是实际账户数据或市场报价。不要在 Notebook 中写入密码、私钥、token 或真实客户明细。

## 1. 安装依赖并定位本目录

在仓库根目录或此目录启动 Jupyter。若当前 Python 环境缺少依赖，请先在终端执行 `pip install -r temp/snowflake-semantic-view/requirements.txt`。

In [ ]:
from pathlib import Path
import os
import sys
import yaml

if Path('position_analytics.yaml').exists():
    BASE_DIR = Path.cwd()
elif Path('temp/snowflake-semantic-view/position_analytics.yaml').exists():
    BASE_DIR = Path('temp/snowflake-semantic-view').resolve()
else:
    raise FileNotFoundError('请从仓库根目录或 temp/snowflake-semantic-view 目录启动 Notebook。')

sys.path.insert(0, str(BASE_DIR / 'scripts'))
from snowflake_common import connect_from_env, qualified_schema

YAML_PATH = BASE_DIR / 'position_analytics.yaml'
TARGET_SCHEMA = qualified_schema(os.environ.get('SV_TARGET_SCHEMA', 'SV_FINANCE_LAB.MART'))
MODEL = yaml.safe_load(YAML_PATH.read_text(encoding='utf-8'))
YAML_TEXT = YAML_PATH.read_text(encoding='utf-8')
print('Definition:', YAML_PATH)
print('Target schema:', TARGET_SCHEMA)
print('Semantic View:', MODEL['name'])
print('Logical tables:', [table['name'] for table in MODEL['tables']])


## 2. 检查连接配置并连接

本地交互式学习可使用已配置的 externalbrowser 认证。自动化环境要用组织批准的非交互认证方式。连接帮助函数只读取环境变量，不打印密码或私钥。

In [ ]:
required_env = ['SNOWFLAKE_ACCOUNT', 'SNOWFLAKE_USER']
missing = [name for name in required_env if not os.environ.get(name)]
if missing:
    raise EnvironmentError('请先配置环境变量：' + ', '.join(missing))
print('Required connection variables are present; values are not displayed.')
conn = connect_from_env()
with conn.cursor() as cur:
    cur.execute('SELECT CURRENT_ACCOUNT(), CURRENT_USER(), CURRENT_ROLE(), CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE()')
    print(cur.fetchone())


## 3. 先执行数据质量检查

如果还没创建教学表，请先在 Snowsight Worksheet 执行 `sql/00_setup_demo.sql`。这份脚本会重建同名练习表，所以 Notebook 不会自动运行它。

In [ ]:
checks = {
    'duplicate_position_keys': f'''SELECT COUNT(*) FROM (
        SELECT PORTFOLIO_ID, SECURITY_ID, AS_OF_DATE
        FROM {TARGET_SCHEMA}.FACT_POSITION_DAILY
        GROUP BY 1,2,3 HAVING COUNT(*) > 1
    )''',
    'orphan_positions': f'''SELECT COUNT(*)
        FROM {TARGET_SCHEMA}.FACT_POSITION_DAILY p
        LEFT JOIN {TARGET_SCHEMA}.DIM_PORTFOLIO f ON p.PORTFOLIO_ID=f.PORTFOLIO_ID
        LEFT JOIN {TARGET_SCHEMA}.DIM_SECURITY s ON p.SECURITY_ID=s.SECURITY_ID
        WHERE f.PORTFOLIO_ID IS NULL OR s.SECURITY_ID IS NULL''',
}
with conn.cursor() as cur:
    for label, sql in checks.items():
        cur.execute(sql)
        result = cur.fetchone()[0]
        print(f'{label}: {result}')
        assert result == 0, f'{label} must be zero before semantic modeling'
print('Data contract checks passed.')


### 环境身份是独立门禁

执行 verify-only 或部署前，先在当前连接运行下面的 SQL，并将结果与本次批准的目标环境记录逐项核对。Notebook/脚本不会自动阻止你连到错误的 Snowflake account；任何一项不匹配都应停止，不要执行部署单元格。

```sql
SELECT CURRENT_ACCOUNT(), CURRENT_REGION(), CURRENT_USER(),
       CURRENT_ROLE(), CURRENT_DATABASE(), CURRENT_SCHEMA(),
       CURRENT_WAREHOUSE();
```

本 Notebook 的部署单元格会执行 CREATE OR ALTER。`DEPLOY_APPROVED = True` 只是手工开关，不是审批系统，也不会自动证明 PR、权限或业务回归已通过。操作契约和当前脚本限制见 `references/workflow-execution-contract.md`。


## 4. 调用 Snowflake verify-only

第三个参数 `TRUE` 表示只验证，不部署对象。该步骤能够发现不少定义问题，但不能证明指标业务口径或自然语言查询一定正确。

In [ ]:
with conn.cursor() as cur:
    cur.execute(
        'CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(%s, %s, TRUE)',
        (TARGET_SCHEMA, YAML_TEXT),
    )
    verify_result = cur.fetchone()
print('verify_only:', verify_result[0] if verify_result else 'OK')


## 5. 可选：部署到当前目标环境

默认不部署。只有在确认账号、schema、对象所有权和数据影响后，才将 `DEPLOY_APPROVED` 手动改为 `True` 并执行该单元格。调用使用 `create_or_alter=TRUE`。在生产环境中，应由 PR 审批和 CI 门禁决定是否允许部署。

In [ ]:
DEPLOY_APPROVED = False  # 明确审查目标环境后才改为 True
if DEPLOY_APPROVED:
    with conn.cursor() as cur:
        cur.execute(
            'CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(%s, %s, FALSE, TRUE)',
            (TARGET_SCHEMA, YAML_TEXT),
        )
        deploy_result = cur.fetchone()
        print('deployment:', deploy_result[0] if deploy_result else 'OK')
        cur.execute(f'DESCRIBE SEMANTIC VIEW {TARGET_SCHEMA}.{MODEL["name"]}')
        description_rows = cur.fetchall()
        print('DESCRIBE row count:', len(description_rows))
else:
    print('Skipped deployment. Set DEPLOY_APPROVED=True only after reviewing the target.')


## 6. 检查语义指标与原始表对账

执行前请确保 Semantic View 已部署。结果应与 SQL/物理表独立聚合一致，并与本演示固定期望值一致。生产测试应使用经业务确认的报表或 golden dataset 作为参考，而不是只依赖这组教学数字。

In [ ]:
from decimal import Decimal

AS_OF_DATE = '2026-10-27'
view_sql = f'{TARGET_SCHEMA}.{MODEL["name"]}'
semantic_query = f'''SELECT PORTFOLIO_NAME, AS_OF_DATE,
       AGG(TOTAL_MARKET_VALUE_USD), AGG(TOTAL_COST_BASIS_USD),
       AGG(TOTAL_UNREALIZED_PNL_USD), AGG(POSITION_COUNT)
FROM {view_sql}
WHERE AS_OF_DATE = %s
GROUP BY PORTFOLIO_NAME, AS_OF_DATE
ORDER BY PORTFOLIO_NAME'''
source_query = f'''SELECT p.PORTFOLIO_NAME, f.AS_OF_DATE,
       SUM(f.MARKET_VALUE_USD), SUM(f.COST_BASIS_USD),
       SUM(f.UNREALIZED_PNL_USD), COUNT(*)
FROM {TARGET_SCHEMA}.FACT_POSITION_DAILY f
JOIN {TARGET_SCHEMA}.DIM_PORTFOLIO p ON p.PORTFOLIO_ID=f.PORTFOLIO_ID
WHERE f.AS_OF_DATE = %s
GROUP BY p.PORTFOLIO_NAME, f.AS_OF_DATE
ORDER BY p.PORTFOLIO_NAME'''

def fetch_rows(sql):
    with conn.cursor() as cur:
        cur.execute(sql, (AS_OF_DATE,))
        return cur.fetchall()

semantic_rows = fetch_rows(semantic_query)
source_rows = fetch_rows(source_query)
assert len(semantic_rows) == len(source_rows), 'Portfolio row count differs'
for semantic_row, source_row in zip(semantic_rows, source_rows):
    assert semantic_row[0] == source_row[0]
    assert str(semantic_row[1]) == str(source_row[1])
    for index in (2, 3, 4):
        assert Decimal(str(semantic_row[index])).quantize(Decimal('0.01')) == Decimal(str(source_row[index])).quantize(Decimal('0.01'))
    assert int(semantic_row[5]) == int(source_row[5])
print('Semantic View and independent source aggregates match.')
for row in semantic_rows:
    print(row)


## 7. 生产化之前还需要补什么？

- 对照投资运营/绩效/风险认可的来源报表验证金额、日期、成本基础、应计利息和币种规则。
- 加入多估值日、缺失证券、重复键、零分母、晚到记录和慢变维度的测试。
- 用低权限角色测试 Semantic View 与 Cortex Agent 的授权边界；不要只用管理员角色。
- 将 YAML、测试集、发布审批、版本 SHA、Query ID 和回滚定义纳入版本控制与部署记录。
- 如果通过 Cortex Analyst/Agent 使用自然语言，再维护 Verified Query Repository / 评估问题集并验证生成 SQL 的业务正确性。

验证完成后关闭连接：

In [ ]:
conn.close()
print('Connection closed.')
